# Gatefall — LoRA training on Google Colab (kohya-ss/sd-scripts, free GPU)

Trains a per-character LoRA against Pony Diffusion V6 XL (or
Illustrious) so future generations of that character stay consistent
across pose/crop/expression, instead of drifting the way a bare
fixed-seed prompt does once composition tokens change.

**Before running:** `Runtime` -> `Change runtime type` -> `T4 GPU` ->
`Save`. Run cells in order.

**Prerequisite — you need a training set first**, not just the single
locked model-sheet image. See `docs/art-direction.md` and the
img2img workflow discussed there: generate 15-20 variants of the
locked character (different angles, expressions, crops) via img2img
at low denoise off the locked reference image, so the outfit/face
stay close across the set. Fresh txt2img rerolls, even at the same
seed, are not consistent enough once the prompt's composition tokens
change — that's the exact problem this LoRA fixes going forward.

**Colab free-tier limits apply** — sessions disconnect after
inactivity and there's a rolling GPU-time cap. LoRA training (unlike
a single image generation) can take 20-60+ minutes depending on
dataset size and epoch count, so budget for that.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi

## 2. Install kohya-ss/sd-scripts

In [ ]:
%cd /content
!git clone https://github.com/kohya-ss/sd-scripts
%cd /content/sd-scripts
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q
!pip install --upgrade -r requirements.txt -q
!pip install accelerate -q

## 3. Mount Google Drive

Used for three things: reading the base checkpoint (reuse the one you
already have in Drive from the ComfyUI notebook, if you put it there),
reading your training image set, and saving the finished LoRA
somewhere that survives the session ending.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Point to the base checkpoint

Train against the **same checkpoint** you generated the training
images with (Pony Diffusion V6 XL, per `docs/art-direction.md`) —
training against a mismatched base checkpoint gives worse results.

In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/gatefall-checkpoints/ponyDiffusionV6XL.safetensors"  # @param {type:"string"}

import os
assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint not found at {CHECKPOINT_PATH} — fix the path (see the ComfyUI notebook's 3a/3b for how you got this file into Drive)."
print("Checkpoint found:", CHECKPOINT_PATH)

## 5. Set up the training image set

1. In Drive, create a folder for this character's training images,
   e.g. `gatefall-lora-training/faelen/`.
2. Put your 15-20 img2img-generated images directly in that folder.
3. For **each** image, add a matching `.txt` caption file with the
   same base filename (e.g. `faelen_01.png` needs `faelen_01.txt`).
   Caption format: a short **trigger word** unique to this character
   (something the base model won't already associate with anything —
   e.g. `flnwarden`), followed by tags for what's *different* in that
   specific image (pose, expression, crop, background) — leave out
   the constant identity traits (hair color, armor, elf ears) that
   are true in every image; the LoRA learns those from the trigger
   word plus the images themselves, not from restating them per file.
   Example `faelen_03.txt`:
   ```
   flnwarden, portrait, upper body, happy expression, slight smile
   ```
4. Update `TRAIN_DATA_DIR` below to match.

In [ ]:
TRAIN_DATA_DIR = "/content/drive/MyDrive/gatefall-lora-training/faelen"  # @param {type:"string"}
TRIGGER_WORD = "flnwarden"  # @param {type:"string"}

import os
images = [f for f in os.listdir(TRAIN_DATA_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))]
captions = [f for f in os.listdir(TRAIN_DATA_DIR) if f.lower().endswith(".txt")]
print(f"Found {len(images)} images and {len(captions)} caption files in {TRAIN_DATA_DIR}")
missing = [f for f in images if os.path.splitext(f)[0] + ".txt" not in captions]
if missing:
    print("WARNING — these images have no matching .txt caption file:")
    for m in missing:
        print(" ", m)
else:
    print("Every image has a matching caption file. Good to proceed.")

Fix any warnings above before continuing — a missing caption file
silently gets skipped or errors out depending on trainer settings, so
it's worth catching now.

## 6. Write the dataset config

In [ ]:
NUM_REPEATS = 10  # @param {type:"integer"}

dataset_toml = f"""
[general]
caption_extension = '.txt'
shuffle_caption = true

[[datasets]]
resolution = 1024
batch_size = 1

  [[datasets.subsets]]
  image_dir = '{TRAIN_DATA_DIR}'
  num_repeats = {NUM_REPEATS}
"""

with open("/content/dataset_config.toml", "w") as f:
    f.write(dataset_toml)

print(dataset_toml)

## 7. Train

`network_train_unet_only` is on because it's recommended for SDXL
LoRA training (per sd-scripts' own SDXL docs). `sdpa` uses PyTorch's
built-in attention so no separate xformers install is needed. This is
the slow cell — 20-60+ minutes depending on dataset size and
`MAX_TRAIN_EPOCHS`; the Colab tab can be left in the background but
don't close it or let the session idle-disconnect.

In [ ]:
OUTPUT_NAME = "faelen_lora"  # @param {type:"string"}
OUTPUT_DIR = "/content/lora_output"  # @param {type:"string"}
MAX_TRAIN_EPOCHS = 10  # @param {type:"integer"}

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd /content/sd-scripts
!accelerate launch --num_cpu_threads_per_process 1 sdxl_train_network.py \
  --pretrained_model_name_or_path="{CHECKPOINT_PATH}" \
  --dataset_config="/content/dataset_config.toml" \
  --output_dir="{OUTPUT_DIR}" \
  --output_name="{OUTPUT_NAME}" \
  --save_model_as=safetensors \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --network_train_unet_only \
  --learning_rate=1e-4 \
  --optimizer_type="AdamW8bit" \
  --lr_scheduler="cosine" \
  --max_train_epochs={MAX_TRAIN_EPOCHS} \
  --save_every_n_epochs=2 \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --cache_text_encoder_outputs \
  --no_half_vae \
  --sdpa

## 8. Save the trained LoRA to Drive (so it survives the session)

In [ ]:
import shutil, glob, os

DRIVE_LORA_DIR = "/content/drive/MyDrive/gatefall-loras"  # @param {type:"string"}
os.makedirs(DRIVE_LORA_DIR, exist_ok=True)

for f in glob.glob(os.path.join(OUTPUT_DIR, "*.safetensors")):
    shutil.copy(f, DRIVE_LORA_DIR)
    print("Saved:", os.path.join(DRIVE_LORA_DIR, os.path.basename(f)))

## 9. Use it in ComfyUI

1. Get the `.safetensors` file from `gatefall-loras/` in Drive into
   `ComfyUI/models/loras/` — in the ComfyUI Colab notebook, mount
   Drive the same way (3b's pattern) and symlink/copy it in, or
   `wget`/copy it directly since it's already in your own Drive.
2. In the ComfyUI graph, add a **`LoraLoader`** node between
   `Load Checkpoint` and the rest of the graph (it takes the
   checkpoint's MODEL/CLIP outputs as input and passes modified
   versions onward — rewire `KSampler` and both `CLIP Text Encode`
   nodes to pull from `LoraLoader`'s outputs instead of
   `Load Checkpoint`'s directly).
3. Select the trained LoRA file in that node, set strength around
   `0.7-0.9` to start.
4. Include the trigger word (`flnwarden` if you kept the default
   above) in your prompt — that's what activates the learned
   character identity.
5. Now pose/crop/expression prompts should hold the design far more
   reliably than the bare fixed-seed approach did.